# SMART PARKING AI

Standalone, image-only parking-space detector. In VS Code, select a Python 3.10?3.12 notebook kernel and choose **Run All**. The first cell installs dependencies and rebuilds the complete app in a `parking_detector` folder beside this notebook. The next cell starts Streamlit and displays the local app link. The first image analysis downloads the pretrained YOLOv8 Nano model.

Upload JPG, JPEG, or PNG parking images in the app. Review automatic space proposals or draw and save a parking layout for repeated viewpoints. No camera or video input is used.


In [ ]:
from pathlib import Path
import base64
import io
import sys
import zipfile
import subprocess

# Rebuild the complete application beside this notebook, so the notebook can be shared alone.
PROJECT_DIR = Path.cwd() / "parking_detector"
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
if not (PROJECT_DIR / "app.py").is_file():
    bundled_sources = """..."""
    with zipfile.ZipFile(io.BytesIO(base64.b64decode(bundled_sources))) as archive:
        archive.extractall(PROJECT_DIR)

print(f"Application files are ready in: {PROJECT_DIR}")
print("Installing the app requirements into this notebook's Python environment...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")],
    check=True,
)
print("Dependencies installed. Run the next cell to start SMART PARKING AI.")


In [ ]:
from pathlib import Path
import os
import socket
import subprocess
import sys
import time
import urllib.error
import urllib.request
from IPython.display import display, Markdown

# Re-running this cell in the same kernel reuses the already running app.
existing_process = globals().get("_parking_server_process")
existing_url = globals().get("_parking_server_url")
if existing_process is not None and existing_process.poll() is None:
    print(f"SMART PARKING AI is already running at {existing_url}")
    display(Markdown(f"[Open SMART PARKING AI]({existing_url})"))
else:
    PROJECT_DIR = Path.cwd() / "parking_detector"
    if not (PROJECT_DIR / "app.py").is_file():
        raise FileNotFoundError("Run the setup cell above first to create the app files.")

    # Find a free local port without disturbing any other app already using 8501.
    port = None
    for candidate in range(8501, 8600):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe:
            try:
                probe.bind(("127.0.0.1", candidate))
                port = candidate
                break
            except OSError:
                continue
    if port is None:
        raise RuntimeError("No available localhost port found between 8501 and 8599.")

    app_url = f"http://127.0.0.1:{port}"
    log_path = PROJECT_DIR / "streamlit.log"
    log_file = log_path.open("a", encoding="utf-8")
    creation_flags = getattr(subprocess, "DETACHED_PROCESS", 0) | getattr(
        subprocess, "CREATE_NEW_PROCESS_GROUP", 0
    )
    try:
        process = subprocess.Popen(
            [
                sys.executable, "-m", "streamlit", "run", "app.py",
                "--server.headless", "true",
                "--server.address", "127.0.0.1",
                "--server.port", str(port),
            ],
            cwd=PROJECT_DIR,
            stdin=subprocess.DEVNULL,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            creationflags=creation_flags,
        )
    finally:
        log_file.close()

    health_url = f"{app_url}/_stcore/health"
    deadline = time.monotonic() + 40
    while time.monotonic() < deadline:
        if process.poll() is not None:
            details = log_path.read_text(encoding="utf-8", errors="replace")[-6000:]
            raise RuntimeError(f"Streamlit exited before it became ready. Log:\n{details}")
        try:
            with urllib.request.urlopen(health_url, timeout=2) as response:
                if response.status == 200:
                    break
        except (urllib.error.URLError, TimeoutError, OSError):
            time.sleep(0.5)
    else:
        details = log_path.read_text(encoding="utf-8", errors="replace")[-6000:]
        raise TimeoutError(f"Streamlit did not become ready. Log:\n{details}")

    _parking_server_process = process
    _parking_server_url = app_url
    print(f"SMART PARKING AI is running at {app_url}")
    print(f"Server log: {log_path}")
    display(Markdown(f"[Open SMART PARKING AI]({app_url})"))
    try:
        import webbrowser
        webbrowser.open(app_url)
    except Exception:
        pass


### Project data

Detection history is saved to `parking_detector/database/parking_system.db`; annotated images are stored in `parking_detector/outputs/`. Keep the generated `parking_detector` folder with this notebook if you want to retain detection history and saved layouts.
